In [85]:
import os
os.environ['JAX_ENABLE_X64'] = '1'
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.2' 

%matplotlib widget
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp

import numpy as np
from skimage.restoration import unwrap_phase
from temgym_core.components import Detector
from temgym_core.utils import wavelength2energy, energy2wavelength
from temgym_core.gaussian import GaussianRayBeta, TaylorExpofAction
from temgym_core.evaluate import evaluate_gaussians_jax_scan, evaluate_gaussians_for, evaluate_gaussians_gpu_kernel_wrapper
from temgym_core.gaussian_taylor import (
    AberratedLens, 
    SigmoidAperture,
    run_to_end,

)
from temgym_core.utils import fibonacci_spiral

jax.config.update("jax_enable_x64", True)

Single Beam Case - Not realistic parameters - We are setting the scale of 1 = 1 angstrom for simplicity for now. 

In [86]:
W = 10e-6

Nx = Ny = 128
aperture_pixel_size_x = W/Nx
aperture_pixel_size_y = W/Ny
input_aperture_grid = Detector(z=0.0, pixel_size=(aperture_pixel_size_x, aperture_pixel_size_y), shape=(Nx, Ny))
coords = input_aperture_grid.coords
X, Y = coords[:,0].reshape(input_aperture_grid.shape), coords[:,1].reshape(input_aperture_grid.shape)
x, y = X[0,:], Y[:,0]
extent = (x[0], x[-1], y[0], y[-1])

voltage = 100e3  # in volts
aperture = SigmoidAperture(radius=W/4, sharpness=1000, z=0.0, t_outside=0.0)
components = (aperture, input_aperture_grid)

In [87]:
wavelength = energy2wavelength(voltage)
k0 = 2 * jnp.pi / wavelength
num_rays = 2**16

key1, key2 = jax.random.PRNGKey(0), jax.random.PRNGKey(1)
num_rays_sqrt = int(jnp.ceil(jnp.sqrt(num_rays)))

rx, ry = fibonacci_spiral(num_rays, radius=W/4)
w0 = 5e-8
q = -1j * (2.0 / (k0 * w0**2))
Q_inv = jnp.array([[q, 0.0], [0.0, q]])
Q_inv = jnp.tile(Q_inv, (num_rays, 1, 1))

aperture_area = 2 * np.pi * (aperture.radius)**2
scale_factor = aperture_area / (w0 ** 2 * num_rays * np.pi)
C0 = jnp.ones(num_rays) * (1.0 * scale_factor + 0.0j)
eta = jnp.full((num_rays, 2), 0.0+0.0j)

voltage = jnp.full((num_rays,), voltage)

S = TaylorExpofAction(
    const=jnp.zeros(num_rays, dtype=jnp.complex128),
    lin=jnp.zeros((num_rays, 2), dtype=jnp.complex128),
    quad=Q_inv,
)
rays_in = GaussianRayBeta(x=rx, 
                         y=ry, 
                         dx=jnp.zeros(num_rays), 
                         dy=jnp.zeros(num_rays), 
                         z=jnp.zeros(num_rays), 
                         pathlength=jnp.zeros(num_rays),
                         _one=jnp.ones(num_rays),
                         C=C0,
                         S=S,
                         voltage=voltage)

rays_in.to_vector()

# Create JIT-ed single-ray and VMAP-ed batched versions without overwriting the original
run_to_end_jit = jax.jit(run_to_end)
run_to_end_vmapped = jax.jit(
    jax.vmap(run_to_end_jit, in_axes=(0, None)),
    static_argnums=(1,),
)

In [ ]:
rays_out = run_to_end_vmapped(rays_in, (aperture,))
input_aperture_image = evaluate_gaussians_gpu_kernel_wrapper(rays_out, input_aperture_grid).block_until_ready()

In [89]:
# du, dv = aperture_pixel_size_x, aperture_pixel_size_y
# diffraction_of_aperture = jnp.fft.ifftshift(jnp.fft.fft2(aperture_image)) * (du * dv) / (1j * wavelength * B) # Very important energy conservation factor
# det_image_in_focus = jnp.abs(diffraction_of_aperture)